# Import libraries

In [1]:
import os
from re import search

# Load settings

In [2]:
if search("ricard", os.uname()[1]):
    exec(open('/Users/ricard/gastrulation_multiome_10x/settings.py').read())
    # exec(open('/Users/ricard/gastrulation_multiome_10x/utils.py').read())
elif search("ebi", os.uname()[1]):
    exec(open('/homes/ricard/gastrulation_multiome_10x/settings.py').read())
    # exec(open('/homes/ricard/gastrulation_multiome_10x/utils.py').read())
else:
    exit("Computer not recognised")

## Define I/O

In [3]:
# io["outdir"] = io["basedir"] + "/..."

## Define options 

In [4]:
opts["samples"] = [
	"E7.5_rep1",
	"E7.5_rep2",
	"E8.5_rep1",
	"E8.5_rep2"
]

# Create anndata object

In [5]:
anndatas = [None for i in range(len(opts["samples"]))]
for i in range(len(opts["samples"])):
    sample = opts["samples"][i]
    anndatas[i] = sc.read_10x_mtx(path=io["basedir"]+"/original/"+sample+"/filtered_feature_bc_matrix", gex_only=True)
    # anndatas[i].obs["sample"] = sample
    anndatas[i].obs["cell"] = sample + "_" + anndatas[i].obs.index
    anndatas[i].obs.set_index("cell", inplace=True, drop=True)
    print(sample)
    print(anndatas[i].shape)

E7.5_rep1
(8933, 32285)
E7.5_rep2
(13182, 32285)
E8.5_rep1
(10903, 32285)
E8.5_rep2
(11411, 32285)


Concatenate anndata experiments

In [6]:
# AnnData.concatenate(*adatas, join='inner', batch_key='batch', batch_categories=None, uns_merge=None, index_unique='-', fill_value=None)
adata = anndata.AnnData.concatenate(*anndatas, join='inner', batch_key=None, index_unique=None)
print(adata.shape)

(44429, 32285)


In [7]:
adata.obs.head()

""
cell
E7.5_rep1_AAACAGCCAAACCCTA-1
E7.5_rep1_AAACAGCCAAACTCAT-1
E7.5_rep1_AAACAGCCACAACCTA-1
E7.5_rep1_AAACAGCCAGGAACTG-1
E7.5_rep1_AAACAGCCATCCTGAA-1


Sanity checks

In [8]:
adata.obs.index.duplicated().sum()

0

In [9]:
adata.var.index.duplicated().sum()

0

## Add existing metadata

Load existing metadata

In [10]:
metadata = pd.read_csv(io["metadata"], sep="\t")
metadata.set_index("cell", inplace=True, drop=True)
metadata.head()

,sample,barcode,archR_cell,nFeature_RNA,nCount_RNA,mtFraction_RNA,pass_rnaQC,celltype.mapped,celltype.score,closest.cell,...,celltype.denoised,TSSEnrichment_atac,ReadsInTSS_atac,PromoterRatio_atac,NucleosomeRatio_atac,nFrags_atac,BlacklistRatio_atac,pass_atacQC,celltype.predicted,stage
cell,,,,,,,,,,,,,,,,,,,,,
E7.5_rep1_AAACAGCCAAACCCTA-1,E7.5_rep1,AAACAGCCAAACCCTA-1,E7.5_rep1#AAACAGCCAAACCCTA-1,8588.0,62088.0,19.757441,True,Visceral_endoderm,0.64,cell_79220,...,Visceral_endoderm,15.130,3782.0,0.212931,1.683401,35225.0,0.017289,True,Visceral_endoderm,E7.5
E7.5_rep1_AAACAGCCAAACTCAT-1,E7.5_rep1,AAACAGCCAAACTCAT-1,E7.5_rep1#AAACAGCCAAACTCAT-1,2929.0,7538.0,16.105068,True,Def._endoderm,0.36,cell_44527,...,Surface_ectoderm,12.593,407.0,0.242666,2.428711,3511.0,0.076616,False,Surface_ectoderm,E7.5
E7.5_rep1_AAACAGCCACAACCTA-1,E7.5_rep1,AAACAGCCACAACCTA-1,NaN,4125.0,12134.0,20.092303,True,ExE_ectoderm,1.00,cell_8101,...,ExE_ectoderm,NaN,NaN,NaN,NaN,NaN,NaN,False,ExE_ectoderm,E7.5
E7.5_rep1_AAACAGCCAGGAACTG-1,E7.5_rep1,AAACAGCCAGGAACTG-1,E7.5_rep1#AAACAGCCAGGAACTG-1,1636.0,3268.0,22.552020,True,Def._endoderm,0.80,cell_82546,...,Def._endoderm,12.400,2173.0,0.175377,1.327706,26061.0,0.024596,True,Def._endoderm,E7.5
E7.5_rep1_AAACAGCCATCCTGAA-1,E7.5_rep1,AAACAGCCATCCTGAA-1,E7.5_rep1#AAACAGCCATCCTGAA-1,2676.0,6033.0,21.631029,True,Nascent_mesoderm,0.88,cell_87810,...,Paraxial_mesoderm,9.989,1589.0,0.153302,1.426056,22048.0,0.013176,True,Paraxial_mesoderm,E7.5


Add to the AnnData

In [11]:
# foo = pd.merge(left=adata.obs, right=metadata, left_on=["cell"], right_on=['cell']).set_index("cell")
foo = pd.merge(left=adata.obs, right=metadata, left_index=True, right_index=True)
assert adata.shape[0] == foo.shape[0]
adata.obs = foo

# Save anndata object

In [12]:
io["outfile"] = io["basedir"]+"/processed/rna/anndata.h5ad"
io["outfile"]

'/hps/nobackup2/research/stegle/users/ricard/gastrulation_multiome_10x/processed/rna/anndata.h5ad'

In [13]:
adata.write_h5ad(io["outfile"])

... storing 'sample' as categorical
... storing 'barcode' as categorical
... storing 'archR_cell' as categorical
... storing 'pass_rnaQC' as categorical
... storing 'celltype.mapped' as categorical
... storing 'closest.cell' as categorical
... storing 'cxds_call' as categorical
... storing 'bcds_call' as categorical
... storing 'hybrid_call' as categorical
... storing 'doublet_call' as categorical
... storing 'celltype.denoised' as categorical
... storing 'celltype.predicted' as categorical
... storing 'stage' as categorical
... storing 'feature_types' as categorical
